Do the prompts differ in whether the AI is exactly correct or not?

In [4]:
import pandas as pd
from statsmodels.stats.contingency_tables import cochrans_q

# Load data
df = pd.read_csv("Output_extraction/ai_grading_final_v3.csv")

# Clean columns
df["answer_key_id"] = df["answer_key_id"].astype(str)
df["prompt_type"] = df["prompt_type"].astype(str)

# Check prompt names
print(df["prompt_type"].unique())

# Binary outcome: exact correct estimate or not
df["correct_estimate"] = (
    df["ai_estimated_mistakes"] == df["true_mistakes"]
).astype(int)

# Create repetition number because answer_key_id is not always unique, but we want every datapoint
df["rep"] = df.groupby(
    ["answer_key_id", "true_mistakes", "prompt_type"]
).cumcount()

# Convert to prompt_type as columns
df_wide = df.pivot_table(
    index=["answer_key_id", "true_mistakes", "rep"],
    columns="prompt_type",
    values="correct_estimate",
    aggfunc="first"
)

print(df_wide.columns.tolist())

# Prompt order
prompt_order = [
    "very_pessimistic",
    "pessimistic",
    "neutral",
    "confident",
    "very_confident"
]

# Keep only rows where all five prompt types are present
df_wide = df_wide[prompt_order].dropna()

# Cochran's Q-test
result = cochrans_q(df_wide)

print(result)

<StringArray>
['very_pessimistic', 'pessimistic', 'neutral', 'confident', 'very_confident']
Length: 5, dtype: str
['confident', 'neutral', 'pessimistic', 'very_confident', 'very_pessimistic']
df          4
pvalue      1.7823892226689482e-06
statistic   32.150363783346805


In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
from itertools import combinations
from statsmodels.stats.multitest import multipletests

posthoc_results = []

for p1, p2 in combinations(prompt_order, 2):
    table = pd.crosstab(df_wide[p1], df_wide[p2])#makes a contengency matrix

    test = mcnemar(table, exact=True)

    posthoc_results.append({
        "comparison": f"{p1} vs {p2}",
        "p_value": test.pvalue
    })

posthoc_df = pd.DataFrame(posthoc_results)

# Holm correction
posthoc_df["p_holm"] = multipletests(
    posthoc_df["p_value"],
    method="holm"
)[1]

print(posthoc_df)

                           comparison   p_value    p_holm
0     very_pessimistic vs pessimistic  0.000606  0.004239
1         very_pessimistic vs neutral  0.865194  1.000000
2       very_pessimistic vs confident  0.001572  0.007861
3  very_pessimistic vs very_confident  0.000193  0.001737
4              pessimistic vs neutral  0.000344  0.002753
5            pessimistic vs confident  0.850953  1.000000
6       pessimistic vs very_confident  0.703284  1.000000
7                neutral vs confident  0.000795  0.004770
8           neutral vs very_confident  0.000057  0.000574
9         confident vs very_confident  0.538227  1.000000
